# 00 — Setup Lakehouse

Creates the medallion schemas (`bronze`, `silver`, `gold`) and the Delta tables
consumed by the rest of the pipeline. Idempotent — safe to re-run.

**Pre-req:** the notebook must be attached to the `lh_fraud` Lakehouse in the
`ws-fi-prod` Fabric workspace, and the cluster identity must hold `Contributor`
on the underlying ADLS Gen2 account.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS lh_fraud.{schema}")
    print(f"schema ready: {schema}")

In [ ]:
# ---------------------------------------------------------------------------
# Bronze tables — raw, append-only, partitioned by ingest date
# ---------------------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.bronze.transactions (
    tx_id              STRING,
    pan_token          STRING,
    amount             DECIMAL(18,2),
    currency           STRING,
    merchant_id        STRING,
    merchant_country   STRING,
    is_card_present    BOOLEAN,
    instrument_type    STRING,
    booking_ts         TIMESTAMP,
    raw_payload        STRING,
    ingest_date        DATE,
    eventhub_offset    STRING
) USING DELTA PARTITIONED BY (ingest_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.bronze.decisions (
    decision_id        STRING,
    tx_id              STRING,
    score              DOUBLE,
    decision           STRING,
    sca_outcome        STRING,
    sca_exemption_code STRING,
    model_version      STRING,
    latency_ms         INT,
    decision_ts        TIMESTAMP,
    raw_payload        STRING,
    ingest_date        DATE
) USING DELTA PARTITIONED BY (ingest_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.bronze.cases (
    case_id            STRING,
    tx_id              STRING,
    fraud_type         STRING,
    confirmed          BOOLEAN,
    confirmed_loss_eur DECIMAL(18,2),
    loss_allocation    STRING,
    analyst_id         STRING,
    opened_ts          TIMESTAMP,
    closed_ts          TIMESTAMP,
    raw_payload        STRING,
    ingest_date        DATE
) USING DELTA PARTITIONED BY (ingest_date)
""")

print("bronze tables ready")

In [ ]:
# ---------------------------------------------------------------------------
# Silver tables — cleansed, deduped, typed
# ---------------------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.silver.transactions (
    tx_id              STRING,
    pan_token          STRING,
    amount_eur         DECIMAL(18,2),
    merchant_id        STRING,
    merchant_country   STRING,
    is_eea             BOOLEAN,
    is_card_present    BOOLEAN,
    instrument_type    STRING,
    channel            STRING,
    issuer_country     STRING,
    booking_ts         TIMESTAMP,
    booking_date       DATE
) USING DELTA PARTITIONED BY (booking_date, issuer_country)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.silver.decisions (
    decision_id        STRING,
    tx_id              STRING,
    score              DOUBLE,
    decision           STRING,
    sca_outcome        STRING,
    sca_exemption_code STRING,
    model_version      STRING,
    latency_ms         INT,
    decision_ts        TIMESTAMP,
    decision_date      DATE
) USING DELTA PARTITIONED BY (decision_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.silver.cases (
    case_id            STRING,
    tx_id              STRING,
    fraud_type         STRING,
    confirmed          BOOLEAN,
    confirmed_loss_eur DECIMAL(18,2),
    loss_allocation    STRING,
    analyst_id         STRING,
    opened_ts          TIMESTAMP,
    closed_ts          TIMESTAMP,
    case_date          DATE
) USING DELTA PARTITIONED BY (case_date)
""")

print("silver tables ready")

In [ ]:
# ---------------------------------------------------------------------------
# Gold marts — aggregated for Power BI / EBA reporter consumption
# ---------------------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.gold.fraud_kpis_daily (
    kpi_date           DATE,
    issuer_country     STRING,
    instrument_type    STRING,
    tx_count           BIGINT,
    tx_value_eur       DECIMAL(18,2),
    decline_count      BIGINT,
    decline_rate       DOUBLE,
    fraud_count        BIGINT,
    fraud_value_eur    DECIMAL(18,2),
    loss_value_eur     DECIMAL(18,2),
    fraud_rate_bps     DOUBLE
) USING DELTA PARTITIONED BY (issuer_country)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.gold.eba_report_q (
    reporting_country  STRING,
    quarter            STRING,
    instrument         STRING,
    channel            STRING,
    sca_exemption      STRING,
    fraud_type         STRING,
    counterparty_geo   STRING,
    loss_bearer        STRING,
    pisp_initiated     BOOLEAN,
    tx_count           BIGINT,
    tx_value_eur       DECIMAL(18,2),
    fraud_count        BIGINT,
    fraud_value_eur    DECIMAL(18,2),
    loss_value_eur     DECIMAL(18,2)
) USING DELTA PARTITIONED BY (reporting_country, quarter)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.gold.psd2_exemption_mix (
    kpi_date           DATE,
    issuer_country     STRING,
    sca_exemption_code STRING,
    tx_count           BIGINT,
    tx_value_eur       DECIMAL(18,2),
    fraud_count        BIGINT,
    fraud_rate_bps     DOUBLE
) USING DELTA PARTITIONED BY (issuer_country)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_fraud.gold.scoring_latency_p99 (
    kpi_date           DATE,
    model_version      STRING,
    p50_ms             DOUBLE,
    p95_ms             DOUBLE,
    p99_ms             DOUBLE,
    sample_count       BIGINT
) USING DELTA
""")

print("gold tables ready")